In [ ]:
import os
import json
import ssl
import httpx
import truststore
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OPENAI_API_KEY is missing from .env"

ssl_context = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)

client = OpenAI(
    api_key=api_key,
    http_client=httpx.Client(verify=ssl_context),
)

print("API key loaded successfully")

In [2]:
import json

VALID_IDS = {
    item["id"]
    for item in SEARCH_DATA["searchCapabilities"]
}

def moderate_query(user_query):
    """Checks whether the search query is safe to process."""
    moderation = client.moderations.create(
        model="omni-moderation-latest",
        input=user_query
    )

    result = moderation.results[0]

    flagged_categories = [
        category
        for category, is_flagged in result.categories.model_dump().items()
        if is_flagged
    ]

    return {
        "is_safe": not result.flagged,
        "flagged_categories": flagged_categories
    }


def search_ups_capability(user_query):
    # Step 1: moderation
    safety = moderate_query(user_query)

    if not safety["is_safe"]:
        return {
            "capability_id": "NO_RESULT",
            "confidence": 1.0,
            "reason": "This search cannot be processed.",
            "moderated": True
        }

    # Step 2: strict intent search
    prompt = f"""
You are a strict intent-search engine for a UPS mobile app.

Your task is to match a user's search ONLY to one capability ID that exists
in the provided capability catalog.

Rules:
1. Understand typos, abbreviations, spelling mistakes, synonyms, and natural language.
2. Return "NO_RESULT" for anything unrelated to the catalog.
3. Never invent a capability ID or feature.
4. Return "NO_RESULT" if confidence is below 0.70.
5. "shop a parcel" may mean "ship a parcel" and should map to SHIP_PACKAGE.
6. "rst pswd" should map to PASSWORD_RESET.
7. "biryani", "movies", and "weather" must return NO_RESULT.

Capability catalog:
{json.dumps(SEARCH_DATA, indent=2)}

User search:
{user_query}
"""

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "ups_search_result",
                "schema": {
                    "type": "object",
                    "properties": {
                        "capability_id": {
                            "type": "string",
                            "enum": sorted(VALID_IDS) + ["NO_RESULT"]
                        },
                        "confidence": {
                            "type": "number",
                            "minimum": 0,
                            "maximum": 1
                        },
                        "reason": {
                            "type": "string"
                        }
                    },
                    "required": ["capability_id", "confidence", "reason"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )

    result = json.loads(response.output_text)

    # Step 3: backend guardrail — do not trust model output blindly
    if (
        result["capability_id"] not in VALID_IDS
        or result["confidence"] < 0.70
    ):
        return {
            "capability_id": "NO_RESULT",
            "confidence": result["confidence"],
            "reason": "No matching UPS mobile app feature found.",
            "moderated": False
        }

    result["moderated"] = False
    return result

API key loaded successfully


NameError: name 'json' is not defined